In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers.optimization import AdamW
from source.version1.data import trainLoader
from source.version1.model import Model
from source.version1.train import trainModel
from source.version1.loss import WeightedLoss

In [3]:
exc = ['bias','LayerNorm.bias','LayerNorm.weight']

In [4]:
def train(subset, rate):
    torch.cuda.empty_cache()
    train, valid = trainLoader(subset)
    model = Model()
    params = list(model.named_parameters())
    groups = []
    groups += [{'params' : [p for n,p in params if not any(ex in n for ex in exc)], 
                'weight_decay':0.001}]
    groups += [{'params' : [p for n,p in params if any(ex in n for ex in exc)], 
                'weight_decay':0.000}]
    model = model.to('cuda:0')
    optimizer = AdamW(groups, lr=rate)
    schedular = ReduceLROnPlateau(optimizer, factor=0.1, min_lr=1e-6, patience=0)
    trainer = {}
    trainer['subset'] = subset
    trainer['model'] = model
    trainer['train'] = train
    trainer['valid'] = valid
    trainer['loss_fn'] = WeightedLoss()
    trainer['optimizer'] = optimizer
    trainer['save'] = '../../model/'
    trainer['epochs'] = 5
    trainer['batch'] = 4
    trainer['schedular'] = schedular
    trainModel(**trainer)
    model = model.cpu()
    del model
    torch.cuda.empty_cache()
    return None

In [ ]:
train(0, 3e-5)

 43%|████▎     | 10724/24728 [05:34<09:02, 25.79it/s, train_loss=2.7173]